<div align="center">

# 🚀 AI ENGINEERING · FROM FIRST API CALL TO INTELLIGENT SYSTEMS

### 01 / Foundations — Your first conversation with an AI model

**Python** · **OpenAI API** · **Prompts** · **Message Roles** · **Secure Configuration**

*An executable, beginner-friendly notebook in an evolving AI Engineering portfolio.*

---

</div>

> **The mission:** Go from a Python notebook to a real API-powered AI interaction. Learn the moving pieces, not just the syntax.

**What you'll build:** a working first API call, a role-based assistant, and a tiny website-summary demonstration using provided sample text.

**Level:** Beginner · **Prerequisites:** Basic Python · **Format:** Learn → Run → Experiment

## 🧭 Learning map

| Step | Concept | Outcome |
|:--|:--|:--|
| 01 | API fundamentals | Explain how your Python code reaches an AI model |
| 02 | Environment setup | Configure dependencies and credentials safely |
| 03 | First model request | Send a message and read the response |
| 04 | Message roles | Distinguish system and user instructions |
| 05 | Prompt design | Turn basic prompts into purposeful tasks |
| 06 | Response inspection | Explore text and usage metadata |
| 07 | Mini project | Summarize website-like content |
| 08 | Practice | Customize, test, and extend |

> **Portfolio note:** This is **Chapter 01: Introduction**. Future chapters can build on this foundation with structured output, tools, RAG, and agents.

## 01 · What happens when you call an AI model?

Your Python application does not run the hosted model locally. It sends a **request** to a provider's API, and the provider sends back a **response**.

```mermaid
flowchart LR
    A[🧑‍💻 Python notebook] --> B[🔐 OpenAI SDK + API key]
    B --> C[☁️ OpenAI API]
    C --> D[🧠 AI model]
    D --> E[📦 Structured API response]
    E --> F[💬 Text displayed in notebook]
```

**Three words to know**

- **API:** A defined interface that lets one application communicate with another.
- **SDK:** A Python library that makes API requests easier to write and manage.
- **Model:** The AI system that processes your messages and produces an output.

> API use may incur charges depending on your account and chosen model. Check the provider's current pricing before experimenting at scale.

## 02 · Prepare your environment

From a terminal **inside your project folder**, install the notebook's dependencies:

```powershell
uv add openai python-dotenv ipykernel
```

If your project is not yet initialized, run `uv init` in the project folder first. Then select its Python environment as the notebook kernel.

Create a file named **`.env`** in the project folder:

```dotenv
OPENAI_API_KEY=your_api_key_here
```

Add the following to **`.gitignore`** *before pushing to GitHub*:

```gitignore
.env
.venv/
__pycache__/
.ipynb_checkpoints/
```

**Security rule:** Never paste your real API key into code cells, screenshots, notebook outputs, or Git commits.

In [1]:
# Core libraries: no API call is made in this cell.
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

print("✅ Imports complete. Ready to configure the client.")

✅ Imports complete. Ready to configure the client.


### 🔒 Load the secret without displaying it

The SDK reads `OPENAI_API_KEY` from the environment. We only check whether a key exists; we **never print its value**.

In [2]:
load_dotenv(override=False)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key or not api_key.strip():
    raise RuntimeError(
        "OPENAI_API_KEY is missing. Add it to your local .env file "
        "and make sure that file is excluded by .gitignore."
    )

client = OpenAI(api_key=api_key)
print("✅ API key loaded; client initialized (no request sent yet).")

✅ API key loaded; client initialized (no request sent yet).


## 03 · Your first AI conversation 🎉

A request needs two important ingredients:

1. A **model identifier** such as `gpt-5-nano`.
2. A list of **messages** that describe the conversation.

The original notebook used the **Chat Completions API**. We keep that interface here so you can see the mechanics clearly.

> **Note:** Model access and availability depend on your API account. If the selected model is unavailable, update the model name to one your project can access.

In [3]:
MODEL = "gpt-5-nano"

first_message = "Hello, GPT! This is my first ever message to you! Hi!"
messages = [{"role": "user", "content": first_message}]

print("📤 Sending:", first_message)
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
)

reply = response.choices[0].message.content
print("\n📥 Model response:")
print(reply or "[The model returned no text content.]")

📤 Sending: Hello, GPT! This is my first ever message to you! Hi!

📥 Model response:
Hi! Welcome, and nice to meet you. I’m glad you reached out.

I can help with lots of things, for example:
- Answer questions and explain concepts
- Help with writing (emails, essays, stories, resumes)
- brainstorm ideas or plan projects
- debugging or understanding code
- summarize articles or books
- learn new topics with quick explanations
- plan trips, schedules, or study plans

If you’re not sure where to start, here are quick prompts you can try:
- Explain [topic] like I’m five
- Help me draft a [type of document]
- Show me a quick summary of [subject/article]
- Debug this code snippet: [paste code]
- Suggest a reading list or learning plan for [topic]

What would you like to do today?


### 🧩 Decode the response

- `response.choices` contains model-generated choice objects.
- `response.choices[0].message.content` accesses the text in the first returned choice.
- `response.usage` can provide token usage information, when available.

The output is **generated at runtime** and can vary between requests; the notebook does not pretend to have predetermined answers.

In [4]:
print("Requested model:", MODEL)
print("Response model:", response.model)
print("Finish reason:", response.choices[0].finish_reason)
if response.usage is not None:
    print("Input tokens:", response.usage.prompt_tokens)
    print("Output tokens:", response.usage.completion_tokens)
    print("Total tokens:", response.usage.total_tokens)
else:
    print("Token usage was not supplied in this response.")

Requested model: gpt-5-nano
Response model: gpt-5-nano-2025-08-07
Finish reason: stop
Input tokens: 21
Output tokens: 812
Total tokens: 833


## 04 · System prompts vs user prompts

The **system message** supplies high-priority guidance about assistant behavior. The **user message** describes the task or question. Together, they shape the model's response.

| Role | Question it answers | Example |
|:--|:--|:--|
| `system` | *How should I behave?* | "Explain clearly to beginners." |
| `user` | *What should I do now?* | "Explain what an API is." |
| `assistant` | *What did the model say?* | Earlier model response in a multi-turn chat |

The model may not obey every instruction perfectly; always review important outputs.

In [5]:
role_messages = [
    {
        "role": "system",
        "content": (
            "You are a patient AI engineering instructor. "
            "Explain concepts simply and include one practical analogy. "
            "Keep the answer to 4 sentences or fewer."
        ),
    },
    {"role": "user", "content": "What is an API, and why do AI engineers use one?"},
]

role_response = client.chat.completions.create(model=MODEL, messages=role_messages)
display(Markdown(role_response.choices[0].message.content or "*No text returned.*"))

An API (Application Programming Interface) is a defined way for software components to talk to each other—specifying what requests you can send and what data formats you’ll get back. AI engineers use APIs to access models, data services, or compute resources without building everything from scratch, and to plug tools into their apps. It’s like a restaurant menu: you place an order (API call) and the kitchen (service) returns the dish (response) without you needing to know how it’s prepared. This lets teams mix and match best-in-class components quickly and safely.

## 05 · Prompt design: instructions + context + output format

A useful prompt often contains **a task**, **relevant input**, and **the expected format**.

**Less specific:** "Summarize this website."

**More specific:** "Summarize the supplied website excerpt in 3 bullets, mention announcements if any, and do not invent details."

We'll now adapt the **website-summary prompt** from the original notebook into a complete, runnable demonstration. This is a **sample text** supplied directly to the model; it is **not** live website scraping.

In [6]:
system_prompt = """
You are a witty website summarizer.
Summarize only the text provided by the user.
Ignore navigation-like material. Use light humor without distorting facts.
Respond in Markdown with 3 concise bullets and an Announcements section.
If no announcement is present, say so. Do not invent information.
""".strip()

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.
""".strip()

sample_website_text = """
Home | About | Contact | Careers
Welcome to Orbit AI Lab, a learning community for aspiring AI engineers.
We teach Python, model APIs, and responsible AI application design.
Announcement: Our free introductory Python workshop opens on December 5.
Footer | Privacy | Terms
""".strip()

website_messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user", "content": f"{user_prompt_prefix}\n\n{sample_website_text}"},
]

website_response = client.chat.completions.create(
    model=MODEL,
    messages=website_messages,
)
display(Markdown(website_response.choices[0].message.content or "*No text returned.*"))

- Orbit AI Lab is a learning community for aspiring AI engineers.
- It teaches Python, model APIs, and responsible AI application design.
- A free introductory Python workshop opens on December 5.

## Announcements
- Our free introductory Python workshop opens on December 5.

## 06 · A reusable function (the first step toward an application)

Instead of rewriting the API call every time, wrap it in a function. That makes your notebook more maintainable and your experiments easier to repeat.

In [7]:
def ask_ai(question: str, *, system_instruction: str = "You are a helpful assistant.") -> str:
    """Send one text question to the chat model and return its reply."""
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    result = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": question},
        ],
    )
    return result.choices[0].message.content or ""

print(ask_ai("Explain the difference between an AI model and an API in one sentence."))

An AI model is the trained algorithm that generates predictions from data, while an API is a software interface that exposes that model’s functionality (via endpoints and requests/responses) so other programs can use it.


## 07 · Hands-on experiments 🧪

Try changing **one thing at a time**:

- **Tone:** Change `system_instruction` from "helpful" to "technical interviewer".
- **Audience:** Ask the same question for a beginner and for a senior developer.
- **Format:** Request two bullets, a table, or a one-sentence answer.
- **Input:** Replace `sample_website_text` with your own short text.

**Mini challenge:** Ask the model to explain *tokens, latency, and temperature* in beginner-friendly language. Then verify the explanations with your own reading.

In [8]:
# ✍️ YOUR TURN: edit these values and rerun the cell.
my_question = "Explain tokens, latency, and temperature to a beginner using one analogy each."
my_style = "You are an AI engineering teacher. Be accurate, concise, and beginner-friendly."

display(Markdown(ask_ai(my_question, system_instruction=my_style)))

Here are three one-idea analogies, one for each concept:

- Tokens: Lego bricks
  - Analogy: Text is built from small bricks called tokens. The model reads and writes with these bricks, not with full words every time. Some words break into multiple bricks, and there’s a limit to how many bricks can be used in a single build (input plus output).
  - Quick takeaway: tokens are the building blocks the model uses to process language.

- Latency: Time you wait after clicking “send”
  - Analogy: When you press generate, the server has to work and send back the result. The longer the journey (network, server load, model complexity), the longer you wait.
  - Quick takeaway: latency is the delay between your request and the model’s response.

- Temperature: Spice level in cooking
  - Analogy: Temperature controls how adventurous the model’s word choices are. Low temperature = bland, predictable answers; high temperature = more creative but riskier and less predictable.
  - Quick takeaway: temperature tunes randomness; 0 is most deterministic, higher values add variety.

## 08 · Key takeaways and what's next

**You can now:**

- [x] Explain the request → model → response flow
- [x] Load API credentials from a local environment file
- [x] Send a chat request from Python
- [x] Use `system` and `user` messages
- [x] Inspect generated text and token usage
- [x] Build and reuse a prompt-driven summarizer

**Next chapters (planned, not implemented here):** structured outputs and validation → embeddings → retrieval-augmented generation (RAG) → tools and function calling → AI agents → production evaluation and deployment.

---

<div align="center">

**Chapter 01 complete — the first step from Python scripts to AI engineering.** 🚀

*Build it. Understand it. Document it. Improve it.*

</div>